# Lab01 - sprint 2
## kNN: прогноз по похожим винам

Ищем вина с ближайшим содержанием спирта и усредняем их оценки.

$$
d(x,x_i)=|x-x_i|,\qquad \hat q(x)=\frac{1}{|N(x)|}\sum_{i\in N(x)}q_i
$$

Если на границе k соседей несколько вин имеют одинаковое расстояние, берём все.
Так результат не зависит от порядка строк. Поэтому соседей может быть больше k.

После первого сравнения расширяем сетку: k = 5, 15, 30, 60, 120, 240, 480, 720, 960, 1440, 2160, 3200, 4158.
Сравниваем по RMSE на прежней validation.
При равных RMSE выбираем большее k. Test не используем.
Сначала запускаем ноутбук подготовки данных.


In [1]:
from pathlib import Path
import sys, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib
# Backend задаёт Jupyter.
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'src'))
from wine_stats.data import load_wines
from wine_stats.split import split_wines
from wine_stats.regression import fit_linear, regression_metrics
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__})
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.2})
from wine_stats.plotting import use_style
use_style()

{'python': '3.12.14', 'numpy': '2.3.5', 'pandas': '2.2.3'}


In [2]:
processed = ROOT / 'data/processed'
meta = json.loads((processed / 'alcohol_quality_split.json').read_text())
for name, info in meta['files'].items():
    assert hashlib.sha256((ROOT / 'data/raw' / name).read_bytes()).hexdigest() == info['sha256']
manifest_path = processed / 'alcohol_quality_split.csv'
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == meta['manifest_sha256']
manifest = pd.read_csv(manifest_path)
df = load_wines(ROOT / 'data/raw')
pd.testing.assert_frame_equal(manifest, split_wines(df, seed=meta['seed'], test_fraction=meta['test_group_fraction']))
outer_train = manifest['split'].eq('train')
train_data = df.loc[outer_train].reset_index(drop=True)
inner = split_wines(train_data, seed=43, test_fraction=0.2)
roles = manifest.copy()
roles.loc[outer_train, 'split'] = inner['split'].replace({'train': 'fit', 'test': 'validation'}).to_numpy()
assert roles.groupby('group_id')['split'].nunique().max() == 1
assert set(roles['split']) == {'fit', 'validation', 'test'}
assert roles.loc[manifest['split'].eq('test'), 'split'].eq('test').all()
roles.to_csv(processed / 'alcohol_quality_validation_split.csv', index=False)
display(pd.crosstab(roles['split'], roles.wine_type))
fit = df.loc[roles['split'].eq('fit')]
validation = df.loc[roles['split'].eq('validation')]

wine_type,red,white
split,,
fit,1024,3134
test,314,988
validation,261,776


In [3]:
from wine_stats.regression import predict_knn
candidates = [5,15,30,60,120,240,480,720,960,1440,2160,3200,len(fit)]
rows=[]
for k in candidates:
    prediction=predict_knn(fit.alcohol,fit.quality,validation.alcohol,k)
    rows.append({'k':k, **regression_metrics(validation.quality,prediction)})
search=pd.DataFrame(rows)
best_k=int(search.sort_values(['RMSE','k'],ascending=[True,False]).iloc[0]['k'])
display(search.round(4))
a,b=fit_linear(fit.alcohol,fit.quality)
predictions={
    'mean':np.full(len(validation),fit.quality.mean()),
    'linear':a+b*validation.alcohol.to_numpy(),
    'knn':predict_knn(fit.alcohol,fit.quality,validation.alcohol,best_k)}
comparison=pd.DataFrame([{'model':name,**regression_metrics(validation.quality,p)} for name,p in predictions.items()])
display(comparison.round(4))
tables=ROOT/'reports/tables'
search.to_csv(tables/'knn_validation_search.csv',index=False)
comparison.to_csv(tables/'knn_validation_comparison.csv',index=False)
print('Выбрано k =',best_k)


,k,MAE,RMSE,R2
0,5,0.6147,0.7690,0.1977
1,15,0.6146,0.7691,0.1977
2,30,0.6142,0.7686,0.1986
3,60,0.6151,0.7684,0.1990
4,120,0.6106,0.7671,0.2018
5,240,0.6073,0.7610,0.2143
6,480,0.6070,0.7590,0.2186
7,720,0.6071,0.7589,0.2187
8,960,0.6089,0.7618,0.2127
9,1440,0.6112,0.7662,0.2036


,model,MAE,RMSE,R2
0,mean,0.6779,0.8586,-0.0000
1,linear,0.6093,0.7676,0.2007
2,knn,0.6071,0.7589,0.2187


Выбрано k = 720


## Lab resume

После расширения сетки минимальная RMSE получена при k = 720: 0,75894 против 0,75899 при k = 480.
Разница около 0,00005 балла - практически ничтожная; уверенного преимущества одного k над другим не установлено.
По заранее выбранному правилу минимума RMSE выбираем k = 720. При k ≥ 960 ошибка начинает расти.
При k = 4158 модель использует всю обучающую часть и совпадает с прогнозом средним.

Линейная модель даёт RMSE = 0,76762. Выигрыш kNN - около 1,1%.
Лучший вариант теперь внутри проверенной сетки.

Значение k выбрано по этой же validation, поэтому её результат может быть оптимистичным.
Итоговую ошибку выбранной модели нужно оценить на отложенном test.

### cat_mark_table

| cat_id | notebook_link | coeff | comment |
|---|---|---|---|
| - | [kNN](02_knn.ipynb) | - | kNN без sklearn, подбор k, сравнение с baseline и МНК. |
